# 🎙️ Proyecto Auditoria DIGI — Fase 1: Procesado de Audio

**Nodos cubiertos:** 1. ffmpeg → 2. DeepFilterNet 3 → 3. Silero VAD → 4. VoxLingua107 (LID)

**Objetivo:** Tomar un archivo de audio de llamada telefónica (.ogg) y entregarlo limpio, recortado y con el idioma identificado, listo para transcribir en la Fase 2.

---
> ⚠️ **Nota de versiones:** Las versiones de librerías abajo son las estables en junio 2026. Fija (pin) siempre las versiones el día del despliegue en producción.

## ✅ Paso 0 — Verificar GPU
Antes de nada, confirmamos que Colab nos asignó GPU.

In [ ]:
import subprocess
resultado = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if resultado.returncode == 0:
    print('✅ GPU disponible:')
    # Mostrar solo la línea con el nombre de la GPU
    for linea in resultado.stdout.split('\n'):
        if 'Tesla' in linea or 'RTX' in linea or 'A100' in linea or 'T4' in linea or 'V100' in linea:
            print(' ', linea.strip())
            break
else:
    print('❌ Sin GPU. Ve a: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU')

## 📦 Paso 1 — Instalar dependencias

Solo se instala lo necesario para esta fase. Tarda ~2 minutos la primera vez.

In [ ]:
# Nodo 1: ffmpeg (ya viene en Colab, pero lo verificamos)
!ffmpeg -version 2>&1 | head -1

# Nodo 2: DeepFilterNet 3 — reducción de ruido
!pip install -q deepfilternet

# Nodo 3: Silero VAD — detección de voz
!pip install -q silero-vad

# Nodo 4: SpeechBrain + VoxLingua107 — identificación de idioma
!pip install -q speechbrain

# Librería de audio usada en todo el pipeline
!pip install -q torchaudio soundfile pydub

print('\n✅ Instalación completada')

## 🎵 Paso 2 — Subir un archivo de audio de prueba

Sube un archivo .ogg, .mp3 o .m4a de una llamada de prueba (puede ser cualquier audio corto).

In [ ]:
from google.colab import files
import os

print('Selecciona tu archivo de audio...')
subidos = files.upload()

ARCHIVO_ORIGINAL = list(subidos.keys())[0]
print(f'\n✅ Archivo recibido: {ARCHIVO_ORIGINAL}')
print(f'   Tamaño: {os.path.getsize(ARCHIVO_ORIGINAL) / 1024:.1f} KB')

## 🔧 Nodo 1 — ffmpeg: Decodificar y normalizar el audio

**¿Qué hace?** Convierte cualquier formato de entrada a WAV mono 16kHz, que es el formato estándar que todos los modelos de IA de audio esperan recibir.

In [ ]:
import subprocess

ARCHIVO_WAV = 'audio_normalizado.wav'

# Convertir a WAV mono 16kHz con normalización de volumen
cmd = [
    'ffmpeg', '-y',
    '-i', ARCHIVO_ORIGINAL,
    '-ac', '1',          # mono (1 canal)
    '-ar', '16000',      # 16kHz (estándar para modelos de voz)
    '-af', 'loudnorm',   # normalización de volumen
    ARCHIVO_WAV
]

resultado = subprocess.run(cmd, capture_output=True, text=True)

if resultado.returncode == 0:
    size_original = os.path.getsize(ARCHIVO_ORIGINAL) / 1024
    size_wav = os.path.getsize(ARCHIVO_WAV) / 1024
    print(f'✅ Nodo 1 completado')
    print(f'   Original: {size_original:.1f} KB  →  WAV: {size_wav:.1f} KB')
    print(f'   Formato: mono, 16kHz, volumen normalizado')
else:
    print(f'❌ Error en ffmpeg:')
    print(resultado.stderr[-500:])

## 🔇 Nodo 2 — DeepFilterNet 3: Reducción de ruido

**¿Qué hace?** Elimina el ruido de fondo típico de las llamadas telefónicas (estática, eco, ruido de oficina). Esto mejora mucho la precisión de Whisper.

In [ ]:
import torch
import torchaudio
from df.enhance import enhance, init_df, load_audio, save_audio

ARCHIVO_LIMPIO = 'audio_sin_ruido.wav'

print('🔇 Cargando DeepFilterNet 3...')
model_df, df_state, _ = init_df()

print('   Procesando audio...')
audio, _ = load_audio(ARCHIVO_WAV, sr=df_state.sr())
audio_limpio = enhance(model_df, df_state, audio)
save_audio(ARCHIVO_LIMPIO, audio_limpio, df_state.sr())

# Liberar memoria GPU
del model_df
torch.cuda.empty_cache()

print(f'✅ Nodo 2 completado — ruido eliminado')
print(f'   Archivo limpio: {os.path.getsize(ARCHIVO_LIMPIO)/1024:.1f} KB')

## ✂️ Nodo 3 — Silero VAD: Detectar y recortar silencios

**¿Qué hace?** Detecta exactamente en qué momentos hay voz humana y en qué momentos hay silencio. Solo conservamos los fragmentos con voz, lo que reduce el tiempo de transcripción y evita que Whisper "alucine" inventando palabras en los silencios.

In [ ]:
import torch
import torchaudio
from silero_vad import load_silero_vad, read_audio, get_speech_timestamps, collect_chunks

ARCHIVO_VAD = 'audio_vad.wav'
SAMPLE_RATE = 16000

print('✂️ Cargando Silero VAD v5...')
model_vad = load_silero_vad()

# Cargar audio limpio
wav = read_audio(ARCHIVO_LIMPIO, sampling_rate=SAMPLE_RATE)

# Detectar fragmentos con voz
timestamps = get_speech_timestamps(
    wav,
    model_vad,
    sampling_rate=SAMPLE_RATE,
    threshold=0.5,          # sensibilidad (0.5 = equilibrado)
    min_speech_duration_ms=250,   # ignorar fragmentos < 250ms
    min_silence_duration_ms=300,  # silencio mínimo entre fragmentos
)

# Unir solo los fragmentos con voz
audio_vad = collect_chunks(timestamps, wav)
torchaudio.save(ARCHIVO_VAD, audio_vad.unsqueeze(0), SAMPLE_RATE)

# Calcular reducción
duracion_original = len(wav) / SAMPLE_RATE
duracion_vad = len(audio_vad) / SAMPLE_RATE
reduccion = (1 - duracion_vad / duracion_original) * 100

# Liberar modelo
del model_vad

print(f'✅ Nodo 3 completado')
print(f'   Fragmentos de voz detectados: {len(timestamps)}')
print(f'   Duración original: {duracion_original:.1f}s  →  Con voz: {duracion_vad:.1f}s')
print(f'   Reducción de silencios: {reduccion:.1f}%')

## 🌍 Nodo 4 — VoxLingua107: Identificar el idioma

**¿Qué hace?** Detecta automáticamente el idioma hablado en la llamada. Si no coincide con el idioma esperado del territorio (español para España), genera un flag de alerta en el reporte.

In [ ]:
import torchaudio
import torch
from speechbrain.pretrained import EncoderClassifier

# Idioma esperado según territorio
IDIOMA_ESPERADO = 'es'  # Español

print('🌍 Cargando VoxLingua107 (ECAPA)...')
classifier = EncoderClassifier.from_hparams(
    source='speechbrain/lang-id-voxlingua107-ecapa',
    savedir='tmp_lid'
)

# Cargar audio procesado por VAD
signal, sample_rate = torchaudio.load(ARCHIVO_VAD)

# Clasificar idioma
prediction = classifier.classify_batch(signal)
idioma_detectado = prediction[3][0].split(':')[0].strip()  # código ISO
confianza = float(prediction[1][0].max().exp()) * 100

# Evaluar resultado
coincide = idioma_detectado == IDIOMA_ESPERADO
flag_idioma = not coincide

# Liberar memoria
del classifier
torch.cuda.empty_cache()

print(f'\n✅ Nodo 4 completado')
print(f'   Idioma detectado: {idioma_detectado} (confianza: {confianza:.1f}%)')
print(f'   Idioma esperado: {IDIOMA_ESPERADO}')
if flag_idioma:
    print(f'   ⚠️ FLAG: el idioma no coincide con el territorio')
else:
    print(f'   ✅ Idioma correcto para el territorio')

## 📋 Resumen final — Fase 1 completada

Recapitulamos el estado del audio y los flags generados, listos para pasar a la Fase 2 (transcripción).

In [ ]:
import json

resultado_fase1 = {
    'archivo_original': ARCHIVO_ORIGINAL,
    'archivo_salida': ARCHIVO_VAD,
    'duracion_original_s': round(duracion_original, 2),
    'duracion_procesada_s': round(duracion_vad, 2),
    'reduccion_silencios_pct': round(reduccion, 1),
    'idioma_detectado': idioma_detectado,
    'idioma_esperado': IDIOMA_ESPERADO,
    'confianza_idioma_pct': round(confianza, 1),
    'flags': {
        'idioma_incorrecto': flag_idioma,
    }
}

print('=' * 55)
print('  FASE 1 — RESULTADO')
print('=' * 55)
for k, v in resultado_fase1.items():
    if k != 'flags':
        print(f'  {k}: {v}')
print()
print('  FLAGS:')
for flag, valor in resultado_fase1['flags'].items():
    estado = '⚠️ ACTIVO' if valor else '✅ OK'
    print(f'    {flag}: {estado}')
print('=' * 55)
print()
print('➡️ Archivo listo para Fase 2 (transcripción):', ARCHIVO_VAD)

# Guardar resultado para pasar a la siguiente fase
with open('resultado_fase1.json', 'w') as f:
    json.dump(resultado_fase1, f, indent=2, ensure_ascii=False)
print('   Metadatos guardados en: resultado_fase1.json')